In [1]:
import torch

from Scripts.make_dataset import clean_split_dataset
from Scripts.evaluation import clear_vram, load_model_for_evaluation, ask_model, evaluate_model
from Scripts.model import prepare_trainer, finetune_model

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Arch List: {torch.cuda.get_arch_list()}")
print(f"Is CUDA working? {torch.cuda.is_available()}")

# Test a simple calculation on the GPU
if torch.cuda.is_available():
    x = torch.randn(1).cuda()
    print("Test calculation performed on GPU is successful")

PyTorch Version: 2.10.0+cu128
CUDA Arch List: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
Is CUDA working? True
Test calculation performed on GPU is successful


In [ ]:
# Step A: Clean dataset, and split original val set into val_data and test_data
train_data, val_data, test_data = clean_split_dataset()

# Step B1: Load the model BEFORE fine-tuning
clear_vram()
tokenizer1, model1 = load_model_for_evaluation(load_peft = False)

Original train size: 6251
Cleaned train size: 5938

Original val/test size: 1147
Cleaned val/test size: 1094
Validation Set Size: 547
Test Set Size: 547


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# Try asking model a sample question
question_no = 9

prompt, response, pred = ask_model(val_data[question_no], tokenizer1, model1)

print("PROMPT:\n")
print(prompt)

print("\nMODEL RESPONSE:\n")
print(response)

print("\nPredicted:", pred)
print("Ground truth:", val_data[question_no]["qa"]["answer"])

PROMPT:

<s>[INST] You are a financial analyst. Perform precise calculations. Always output the result as a single number or percentage in the FINAL ANSWER section.


Example 1:
Question: What is the gross profit margin?
CALCULATION: 200 / 500 = 0.4.
FINAL ANSWER: 40.0%

Example 2:
Question: What was the percentage growth in assets?
CALCULATION: (1240 - 1000) / 1000 = 240 / 1000 = 0.24.
FINAL ANSWER: 24.0%

Example 3:
Question: What is the ratio of debt to equity?
CALCULATION: 150 / 400 = 0.375.
FINAL ANSWER: 37.5%
---
    
Now analyze this:
Context: other items on our consolidated financial statements have been appropriately adjusted from the amounts provided in the earnings release , including a reduction of our full year 2016 gross profit and income from operations by $ 2.9 million , and a reduction of net income by $ 1.7 million. . ( 1 ) working capital is defined as current assets minus current liabilities. .
Table: ( in thousands ) | at december 31 , 2016 | at december 31 , 2015 

In [ ]:
# Step B2: Evaluate accuracy of the model BEFORE fine-tuning
evaluate_model(test_data, tokenizer1, model1, n=547)

0 99.9985% 0.995
Current Accuracy: 100.00%
1 10.3% 0.09699999999999999
Current Accuracy: 100.00%
2 6192 -76.0
Current Accuracy: 66.67%
3 6.166666666666667 5.4
Current Accuracy: 50.00%
4 14.8% 0.32
Current Accuracy: 40.00%
5 6.2 6.25
Current Accuracy: 50.00%
6 1.47% 0.02
Current Accuracy: 57.14%
7 286353 28597891.58
Current Accuracy: 50.00%
8 10.94% 0.0942
Current Accuracy: 44.44%
9 1.834 0.182
Current Accuracy: 40.00%
10 -7.4% -0.05
Current Accuracy: 36.36%
11 15.1% 0.15
Current Accuracy: 41.67%
12 0.061 0.15
Current Accuracy: 38.46%
13 53464250 53382250.0
Current Accuracy: 42.86%
14 23.33% 0.23149999999999998
Current Accuracy: 46.67%
15 17.0 17.0
Current Accuracy: 50.00%
16 708343 16609.0
Current Accuracy: 47.06%
17 10.3% 0.10400000000000001
Current Accuracy: 50.00%
18 1379 797.7
Current Accuracy: 47.37%
19 42 0.0241
Current Accuracy: 45.00%
20 183.78% 1.57
Current Accuracy: 42.86%
21 11.54% 0.12
Current Accuracy: 45.45%
22 134.0% 1.33
Current Accuracy: 47.83%
23 320 0.181000000000000

In [ ]:
# Step C1: Prepare the model trainer/settings
tokenizer, model, trainer = prepare_trainer(train_data, val_data)

Map:   0%|          | 0/5938 [00:00<?, ? examples/s]

Map:   0%|          | 0/547 [00:00<?, ? examples/s]

Map:   0%|          | 0/5938 [00:00<?, ? examples/s]

Map:   0%|          | 0/547 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


In [ ]:
# Step C2: Fine tune the model
finetune_model(tokenizer, model, trainer)

Step,Training Loss,Validation Loss
300,0.922000,1.058075
600,0.727200,0.982157
900,0.422400,0.996354
1200,0.345800,0.995456


('./FinQA-LoRA-Adaptor\\tokenizer_config.json',
 './FinQA-LoRA-Adaptor\\special_tokens_map.json',
 './FinQA-LoRA-Adaptor\\chat_template.jinja',
 './FinQA-LoRA-Adaptor\\tokenizer.model',
 './FinQA-LoRA-Adaptor\\added_tokens.json',
 './FinQA-LoRA-Adaptor\\tokenizer.json')

In [ ]:
# Step D1: Load the model AFTER fine-tuning
clear_vram()
tokenizer2, model2 = load_model_for_evaluation(load_peft = True)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# Step D2: Evaluate accuracy of the model AFTER fine-tuning
evaluate_model(test_data, tokenizer2, model2, n=547)

0 0.9954444444444444 0.995
Current Accuracy: 100.00%
1 0.08904444444444444 0.09699999999999999
Current Accuracy: 100.00%
2 -76.0 -76.0
Current Accuracy: 100.00%
3 5.3666666666666664 5.4
Current Accuracy: 100.00%
4 11.2 0.32
Current Accuracy: 80.00%
5 6.25 6.25
Current Accuracy: 83.33%
6 0.016666666666666667 0.02
Current Accuracy: 85.71%
7 28618317.1 28597891.58
Current Accuracy: 87.50%
8 0.09428571428571429 0.0942
Current Accuracy: 88.89%
9 1181.8181818181818 0.182
Current Accuracy: 80.00%
10 -0.05474423606797754 -0.05
Current Accuracy: 81.82%
11 0.15328413134328354 0.15
Current Accuracy: 83.33%
12 0.1489244472844244 0.15
Current Accuracy: 84.62%
13 52506500.0 53382250.0
Current Accuracy: 78.57%
14 0.23264814814814815 0.23149999999999998
Current Accuracy: 80.00%
15 17.0 17.0
Current Accuracy: 81.25%
16 16609.0 16609.0
Current Accuracy: 82.35%
17 0.1036111111111111 0.10400000000000001
Current Accuracy: 83.33%
18 797.6666666666667 797.7
Current Accuracy: 84.21%
19 0.02404404404404404 0.0